In [1]:
# import necessary libraries
import pandas as pd
import numpy as np
import ollama
import json
from tqdm import tqdm 

In [ ]:
# Get clean test set
#test_df = pd.read_csv("clean_test_set.csv")

# Get a sample of the test set for LLM evaluation
#llm_test_sample = test_df.sample(n=200, random_state=42).reset_index(drop=True)
#target_cols = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']

# Save test sample for fairness evaluation for BERT an LLM
#llm_test_sample.to_csv("mini_test_sample.csv", index=False)

In [5]:
llm_test_sample = pd.read_csv("mini_test_sample.csv")
target_cols = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']

In [2]:
# Zero-shot predictions using Ollama
def get_zero_shot_prediction(comment_text, model_name="llama3.1:8b"):
    # Modelin kafasını karıştırmayacak, kesin ve katı kuralları olan sistem ve kullanıcı promptu
    system_prompt = (
        "You are an expert content moderation AI. Analyze the given text and classify it into 6 toxicity categories. "
        "The categories are: toxic, severe_toxic, obscene, threat, insult, and identity_hate. "
        "You must respond ONLY with a raw JSON object containing the category names as keys and binary integers (0 or 1) as values. "
        "Do not include any explanation, no markdown formatting, no code block backticks, just the raw JSON."
    )
    
    user_prompt = f"Text to analyze: '{comment_text}'"
    
    try:
        response = ollama.generate(
            model=model_name,
            system=system_prompt,
            prompt=user_prompt,
            options={"temperature": 0.0} # Kararlılık ve tutarlılık için yaratıcılığı sıfırlıyoruz
        )
        
        # Gelen metni temizleyip JSON olarak parse ediyoruz
        raw_output = response['response'].strip()
        
        # Bazen modeller inatla ```json ... ``` bloğu içinde dönebilir, onu temizleyelim
        if raw_output.startswith("```"):
            raw_output = raw_output.replace("```json", "").replace("```", "").strip()
            
        prediction_dict = json.loads(raw_output)
        
        # Sözlükteki sıralı değerleri tıpkı veri setimizdeki gibi array formatına getiriyoruz
        return [
            prediction_dict.get('toxic', 0),
            prediction_dict.get('severe_toxic', 0),
            prediction_dict.get('obscene', 0),
            prediction_dict.get('threat', 0),
            prediction_dict.get('insult', 0),
            prediction_dict.get('identity_hate', 0)
        ]
    except Exception as e:
        # Eğer LLM beklenmedik bir format döner veya kilitlenirse güvenli liman olarak hepsine 0 diyoruz
        return [0, 0, 0, 0, 0, 0]

In [6]:
# Zero-shot prediction loop
if __name__ == '__main__':
    llm_predictions = []
    
    print("--- Starting Local LLM Zero-Shot Inference ---")
    # tqdm ile işlem sürerken ekranda bir ilerleme çubuğu göreceğiz
    for idx, row in tqdm(llm_test_sample.iterrows(), total=len(llm_test_sample)):
        pred = get_zero_shot_prediction(row['cleaned_text'], model_name="llama3.1:8b")
        llm_predictions.append(pred)
        
    llm_predictions = np.array(llm_predictions)
    true_labels = llm_test_sample[target_cols].values
    
    # --- METRİK HESAPLAMA (Tıpkı DistilBERT'teki gibi İngilizce Format) ---
    from sklearn.metrics import precision_score, recall_score, f1_score
    
    macro_precision = precision_score(true_labels, llm_predictions, average='macro', zero_division=0)
    macro_recall = recall_score(true_labels, llm_predictions, average='macro', zero_division=0)
    macro_f1 = f1_score(true_labels, llm_predictions, average='macro', zero_division=0)
    
    micro_precision = precision_score(true_labels, llm_predictions, average='micro', zero_division=0)
    micro_recall = recall_score(true_labels, llm_predictions, average='micro', zero_division=0)
    micro_f1 = f1_score(true_labels, llm_predictions, average='micro', zero_division=0)
    
    print("\n==================== LOCAL LLM ZERO-SHOT REPORT ====================")
    print(f"Macro -> Precision: {macro_precision:.4f} | Recall: {macro_recall:.4f} | F1-Score: {macro_f1:.4f}")
    print(f"Micro -> Precision: {micro_precision:.4f} | Recall: {micro_recall:.4f} | F1-Score: {micro_f1:.4f}")
    print("====================================================================")

--- Starting Local LLM Zero-Shot Inference ---


100%|██████████| 200/200 [28:58<00:00,  8.69s/it]



==================== LOCAL LLM ZERO-SHOT REPORT ====================
Macro -> Precision: 0.2665 | Recall: 0.6598 | F1-Score: 0.3448
Micro -> Precision: 0.3276 | Recall: 0.8807 | F1-Score: 0.4776


In [10]:
# Few-shot predictions using Ollama
def get_few_shot_prediction(comment_text, model_name="llama3.1:8b"):
    # Sisteme gerçek dünya örnekleri (Few-shot) enjekte ediyoruz
    system_prompt = (
    "You are an expert content moderation AI. Analyze the given text and classify it into 6 toxicity categories: "
    "toxic, severe_toxic, obscene, threat, insult, and identity_hate.\n\n"
    "Carefully study these real-world examples from the dataset to learn the boundaries of each category:\n\n"
    
    "Example 1 (Clean):\n"
    "Text: 'then answer my question why?'\n"
    "Response: {\"toxic\": 0, \"severe_toxic\": 0, \"obscene\": 0, \"threat\": 0, \"insult\": 0, \"identity_hate\": 0}\n\n"
    
    "Example 2 (Toxic):\n"
    "Text: 'Why are you guys reverting my edits? Im trying to cooperate with a picture of a native cherokee wedding and you revert it, i dont understand. How about some workout fatty.'\n"
    "Response: {\"toxic\": 1, \"severe_toxic\": 0, \"obscene\": 0, \"threat\": 0, \"insult\": 0, \"identity_hate\": 0}\n\n"
    
    "Example 3 (Severe Toxic):\n"
    "Text: 'Go fuckin' hang yourself! Fuckin' scum of the earth! Fuck you asshole!'\n"
    "Response: {\"toxic\": 1, \"severe_toxic\": 1, \"obscene\": 1, \"threat\": 0, \"insult\": 1, \"identity_hate\": 0}\n\n"
    
    "Example 4 (Obscene):\n"
    "Text: 'Who is Dreadstar and How is it that they are able to censor me? So I log into Wikipedia one evening and find a few things worth commenting on. Suddenly all my edits are blocked by Dreadstar. WTF and who the heck is Dreadstar?'\n"
    "Response: {\"toxic\": 0, \"severe_toxic\": 0, \"obscene\": 1, \"threat\": 0, \"insult\": 0, \"identity_hate\": 0}\n\n"
    
    "Example 5 (Threat):\n"
    "Text: 'YOU ARE A FUCKING NERD. GO KILL YOURSELF YOU NERD.'\n"
    "Response: {\"toxic\": 1, \"severe_toxic\": 1, \"obscene\": 1, \"threat\": 1, \"insult\": 1, \"identity_hate\": 0}\n\n"
    
    "Example 6 (Insult):\n"
    "Text: 'Aquaman the Big Stupid Loser This article needs to go into more detail about Aquaman being a loser. I don't care what some fanboys and some dude who traces for a living say how powerful he is. Most people know Aquaman as the loser from the Superfriends who talks to fish.'\n"
    "Response: {\"toxic\": 0, \"severe_toxic\": 0, \"obscene\": 0, \"threat\": 0, \"insult\": 1, \"identity_hate\": 0}\n\n"
    
    "Example 7 (Identity Hate):\n"
    "Text: 'All russian are so dumm like you? I mean, you're the best of them. So much to call russia a nation...more like rabbits...'\n"
    "Response: {\"toxic\": 1, \"severe_toxic\": 0, \"obscene\": 0, \"threat\": 0, \"insult\": 1, \"identity_hate\": 1}\n\n"
    
    "You must respond ONLY with a raw JSON object containing the category names as keys and binary integers (0 or 1) as values. Do not include any explanation."
)
    
    user_prompt = f"Text to analyze: '{comment_text}'"
    
    # Geri kalan ollama.generate ve JSON parse adımları Zero-shot fonksiyonu ile tıpatıp AYNI olacak...

    try:
        response = ollama.generate(
            model=model_name,
            system=system_prompt,
            prompt=user_prompt,
            options={"temperature": 0.0} # Kararlılık ve tutarlılık için yaratıcılığı sıfırlıyoruz
        )
        
        # Gelen metni temizleyip JSON olarak parse ediyoruz
        raw_output = response['response'].strip()
        
        # Bazen modeller inatla ```json ... ``` bloğu içinde dönebilir, onu temizleyelim
        if raw_output.startswith("```"):
            raw_output = raw_output.replace("```json", "").replace("```", "").strip()
            
        prediction_dict = json.loads(raw_output)
        
        # Sözlükteki sıralı değerleri tıpkı veri setimizdeki gibi array formatına getiriyoruz
        return [
            prediction_dict.get('toxic', 0),
            prediction_dict.get('severe_toxic', 0),
            prediction_dict.get('obscene', 0),
            prediction_dict.get('threat', 0),
            prediction_dict.get('insult', 0),
            prediction_dict.get('identity_hate', 0)
        ]
    except Exception as e:
        # Eğer LLM beklenmedik bir format döner veya kilitlenirse güvenli liman olarak hepsine 0 diyoruz
        return [0, 0, 0, 0, 0, 0]

In [11]:
# Few-shot prediction loop
if __name__ == '__main__':
    llm_predictions = []
    
    print("--- Starting Local LLM Few-Shot Inference ---")
    # tqdm ile işlem sürerken ekranda bir ilerleme çubuğu göreceğiz
    for idx, row in tqdm(llm_test_sample.iterrows(), total=len(llm_test_sample)):
        pred = get_few_shot_prediction(row['cleaned_text'], model_name="llama3.1:8b")
        llm_predictions.append(pred)
        
    llm_predictions = np.array(llm_predictions)
    true_labels = llm_test_sample[target_cols].values
    
    # --- METRİK HESAPLAMA (Tıpkı DistilBERT'teki gibi İngilizce Format) ---
    from sklearn.metrics import precision_score, recall_score, f1_score
    
    macro_precision = precision_score(true_labels, llm_predictions, average='macro', zero_division=0)
    macro_recall = recall_score(true_labels, llm_predictions, average='macro', zero_division=0)
    macro_f1 = f1_score(true_labels, llm_predictions, average='macro', zero_division=0)
    
    micro_precision = precision_score(true_labels, llm_predictions, average='micro', zero_division=0)
    micro_recall = recall_score(true_labels, llm_predictions, average='micro', zero_division=0)
    micro_f1 = f1_score(true_labels, llm_predictions, average='micro', zero_division=0)
    
    print("\n==================== LOCAL LLM FEW-SHOT REPORT ====================")
    print(f"Macro -> Precision: {macro_precision:.4f} | Recall: {macro_recall:.4f} | F1-Score: {macro_f1:.4f}")
    print(f"Micro -> Precision: {micro_precision:.4f} | Recall: {micro_recall:.4f} | F1-Score: {micro_f1:.4f}")
    print("====================================================================")

--- Starting Local LLM Few-Shot Inference ---


100%|██████████| 200/200 [19:43<00:00,  5.92s/it]


==================== LOCAL LLM FEW-SHOT REPORT ====================
Macro -> Precision: 0.2560 | Recall: 0.5345 | F1-Score: 0.3262
Micro -> Precision: 0.3520 | Recall: 0.8073 | F1-Score: 0.4903
